In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("webadvisor/real-time-anomaly-detection-in-cctv-surveillance")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance


In [2]:
import os

video_dir = "/kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data"

videos = os.listdir(video_dir)
print("Total videos:", len(videos))
print(videos[:10])

Total videos: 16
['roadaccidents', 'assault', 'vandalism', 'arrest', 'shooting', 'arson', 'explosion', 'normal', 'shoplifting', 'robbery']


In [3]:
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data/train.csv")
df.head()

,Unnamed: 0,label,video_name
0,1229,normal,data\normal\Normal_Videos_196_x264.mp4
1,551,normal,data\normal\Normal_Videos179_x264.mp4
2,715,normal,data\normal\Normal_Videos361_x264.mp4
3,1366,roadaccidents,data\roadaccidents\RoadAccidents017_x264.mp4
4,501,normal,data\normal\Normal_Videos125_x264.mp4


In [4]:
video_dir = "/kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data"

video_paths = []
labels = []

for label in os.listdir(video_dir):
    class_path = os.path.join(video_dir, label)
    
    if not os.path.isdir(class_path):
        continue
        
    for file in os.listdir(class_path):
        full_path = os.path.join(class_path, file)
        
        if file.lower().endswith(('.mp4', '.avi', '.mov')):
            video_paths.append(full_path)
            labels.append(label)

print("Total videos found:", len(video_paths))
print(video_paths[:5])

Total videos found: 1900
['/kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data/roadaccidents/RoadAccidents111_x264.mp4', '/kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data/roadaccidents/RoadAccidents009_x264.mp4', '/kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data/roadaccidents/RoadAccidents033_x264.mp4', '/kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data/roadaccidents/RoadAccidents079_x264.mp4', '/kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data/roadaccidents/RoadAccidents006_x264.mp4']


In [5]:
import cv2

sample = video_paths[0]

cap = cv2.VideoCapture(sample)
print("Opened:", cap.isOpened())

ret, frame = cap.read()
print("Frame read:", ret)

cap.release()

Opened: True
Frame read: True


In [6]:
print("Total videos before cleaning:", len(video_paths))


Total videos before cleaning: 1900


In [7]:
# def get_frame_count(path):
#     cap = cv2.VideoCapture(path)
#     count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#     cap.release()
#     return count

# filtered_videos = []
# for v in clean_videos:
#     path = os.path.join(video_dir, v)
#     if get_frame_count(path) > 30:   # threshold
#         filtered_videos.append(v)

In [8]:
import cv2

def has_min_frames(path, min_frames=20):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return total >= min_frames

In [9]:
clean_video_paths = []
clean_labels = []

for path, label in zip(video_paths, labels):
    if has_min_frames(path):
        clean_video_paths.append(path)
        clean_labels.append(label)

print("After cleaning:", len(clean_video_paths))

After cleaning: 1900


In [10]:
print(clean_video_paths[:3])
print(clean_labels[:3])

['/kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data/roadaccidents/RoadAccidents111_x264.mp4', '/kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data/roadaccidents/RoadAccidents009_x264.mp4', '/kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data/roadaccidents/RoadAccidents033_x264.mp4']
['roadaccidents', 'roadaccidents', 'roadaccidents']


In [11]:
def is_too_long(path, max_frames=1000):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return total > max_frames

In [12]:
import pandas as pd

df = pd.DataFrame({
    "video_path": clean_video_paths,
    "label_name": clean_labels
})

df.head()

,video_path,label_name
0,/kaggle/input/datasets/webadvisor/real-time-an...,roadaccidents
1,/kaggle/input/datasets/webadvisor/real-time-an...,roadaccidents
2,/kaggle/input/datasets/webadvisor/real-time-an...,roadaccidents
3,/kaggle/input/datasets/webadvisor/real-time-an...,roadaccidents
4,/kaggle/input/datasets/webadvisor/real-time-an...,roadaccidents


In [13]:
df["label"] = df["label_name"].apply(lambda x: 0 if x == "normal" else 1)

In [14]:
from collections import Counter
print(Counter(df["label"]))

Counter({1: 950, 0: 950})


In [15]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)
train_df = train_df.sample(300)
val_df = val_df.sample(100)

In [16]:
class_counts = train_df["label"].value_counts().to_dict()
print(class_counts)

{1: 151, 0: 149}


In [17]:
weights = train_df["label"].apply(lambda x: 1 / class_counts[x]).values

In [18]:
print("Train size:", len(train_df))
print("Val size:", len(val_df))

print("Train distribution:")
print(train_df["label"].value_counts())

print("Val distribution:")
print(val_df["label"].value_counts())

Train size: 300
Val size: 100
Train distribution:
label
1    151
0    149
Name: count, dtype: int64
Val distribution:
label
0    51
1    49
Name: count, dtype: int64


In [19]:
import numpy as np
import cv2

def extract_frames(video_path, num_frames=16):
    cap = cv2.VideoCapture(video_path)
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames < num_frames:
        cap.release()
        return None
    
    indices = np.linspace(0, total_frames-1, num_frames).astype(int)
    
    frames = []
    for i in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        
        if not ret:
            continue
        
        frame = cv2.resize(frame, (116, 116))
        frames.append(frame)
    
    cap.release()
    
    if len(frames) != num_frames:
        return None
    
    return np.array(frames)

In [20]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [21]:
import torch
from torch.utils.data import Dataset

class VideoDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        while True:
            row = self.df.iloc[idx]
            frames = extract_frames(row["video_path"])
            
            if frames is not None:
                break
            idx = (idx + 1) % len(self.df)
        
        frames = [self.transform(f) for f in frames]
        frames = torch.stack(frames)        # (T, C, H, W)
        frames = frames.permute(1, 0, 2, 3) # (C, T, H, W)
        
        label = torch.tensor(row["label"]).float()
        return frames, label

In [22]:
from torch.utils.data import DataLoader

train_dataset = VideoDataset(train_df, train_transform)
val_dataset = VideoDataset(val_df, val_transform)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)

In [23]:
import torchvision.models.video as models
import torch.nn as nn

from torchvision.models.video import r3d_18, R3D_18_Weights

model = r3d_18(weights=R3D_18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100%|██████████| 127M/127M [00:00<00:00, 218MB/s]


In [24]:
model.fc = nn.Linear(model.fc.in_features, 1)

In [25]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [26]:
# freeze most layers
for param in model.parameters():
    param.requires_grad = False

# unfreeze last block + fc
for param in list(model.layer4.parameters()):
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

In [27]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
import torch.nn as nn

criterion = nn.BCEWithLogitsLoss()

In [28]:
from tqdm import tqdm

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    scheduler.step()
    for videos, labels in tqdm(loader):
        videos = videos.to(device)
        labels = labels.to(device)
        
        outputs = model(videos).squeeze()
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

In [29]:
from sklearn.metrics import f1_score

def validate(model, loader):
    model.eval()
    
    preds = []
    true = []
    
    with torch.no_grad():
        for videos, labels in loader:
            videos = videos.to(device)
            
            outputs = model(videos).squeeze()
            probs = torch.sigmoid(outputs)
            
            pred = (probs > 0.5).cpu().numpy()
            
            preds.extend(pred)
            true.extend(labels.numpy())
    
    return f1_score(true, preds)

In [30]:
best_f1 = 0
EPOCHS=7
for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader)
    val_f1 = validate(model, val_loader)
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "best_model.pth")
        print("✅ Saved best model")

/tmp/ipykernel_23/1845913449.py:6: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
100%|██████████| 150/150 [03:18<00:00,  1.32s/it]


✅ Saved best model


100%|██████████| 150/150 [02:48<00:00,  1.12s/it]


✅ Saved best model


100%|██████████| 150/150 [02:47<00:00,  1.12s/it]


✅ Saved best model


100%|██████████| 150/150 [02:48<00:00,  1.13s/it]


In [31]:
import torch
import torch.nn as nn
from torchvision.models.video import r3d_18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = r3d_18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 1)

model.load_state_dict(torch.load("/kaggle/working/best_model.pth", map_location=device))
model = model.to(device)
model.eval()

VideoResNet(
  (stem): BasicStem(
    (0): Conv3d(3, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3), bias=False)
    (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (conv2): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (relu): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1):

In [32]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [33]:
import cv2
import numpy as np

def extract_frames(video_path, num_frames=6):
    cap = cv2.VideoCapture(video_path)
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames < num_frames:
        cap.release()
        return None
    
    indices = np.linspace(0, total_frames-1, num_frames).astype(int)
    
    frames = []
    for i in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        
        if not ret:
            continue
        
        frame = cv2.resize(frame, (96, 96))
        frames.append(frame)
    
    cap.release()
    
    if len(frames) != num_frames:
        return None
    
    return frames

In [34]:
def predict_video(video_path):
    frames = extract_frames(video_path)
    
    if frames is None:
        return "Skipped"
    
    frames = [transform(f) for f in frames]
    frames = torch.stack(frames)              # (T, C, H, W)
    frames = frames.permute(1, 0, 2, 3)       # (C, T, H, W)
    frames = frames.unsqueeze(0).to(device)   # (1, C, T, H, W)
    
    with torch.no_grad():
        output = model(frames).squeeze()
        prob = torch.sigmoid(output).item()
    
    score = int(prob * 10)
    
    if score <= 3:
        label = "Normal"
    elif score <= 6:
        label = "Suspicious"
    else:
        label = "Anomaly 🚨"
    
    return score, label

In [35]:
import os

test_dir = "/kaggle/input/your-test-folder"

for root, dirs, files in os.walk(test_dir):
    for file in files:
        if file.endswith((".mp4", ".avi", ".mov")):
            path = os.path.join(root, file)
            
            result = predict_video(path)
            print(f"{file} → {result}")

In [36]:
for i in range(5):
    predict_video(val_df.iloc[i]["video_path"])

In [37]:
import cv2
import torch
import numpy as np

def live_anomaly_detection(threshold=0.4):
    cap = cv2.VideoCapture(0)  # 0 = webcam
    
    buffer = []
    num_frames = 16
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_resized = cv2.resize(frame, (116, 116))
        buffer.append(frame_resized)
        
        # When buffer is full → run model
        if len(buffer) == num_frames:
            
            frames = [val_transform(f) for f in buffer]
            frames = torch.stack(frames)
            frames = frames.permute(1, 0, 2, 3)
            frames = frames.unsqueeze(0).to(device)
            
            with torch.no_grad():
                output = model(frames)
                prob = torch.sigmoid(output).item()
            
            score = int(prob * 10)
            is_anomaly = prob > threshold
            
            if not is_anomaly:
                text = f"Normal ({score}/10)"
            elif score <= 6:
                text = f"Suspicious ({score}/10)"
            else:
                text = f"ANOMALY ({score}/10)"
            
            # Slide window (important)
            buffer = buffer[8:]
        
        else:
            text = "Collecting frames..."
        
        # Display result
        cv2.putText(frame, text, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1,
                    (0, 255, 0), 2)
        
        cv2.imshow("Live Anomaly Detection", frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()

In [38]:
import cv2
import torch

def simulate_live(video_path, threshold=0.4):
    cap = cv2.VideoCapture(video_path)
    
    buffer = []
    num_frames = 16
    
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame = cv2.resize(frame, (116, 116))
        buffer.append(frame)
        frame_count += 1
        
        if len(buffer) == num_frames:
            
            frames = [val_transform(f) for f in buffer]
            frames = torch.stack(frames)
            frames = frames.permute(1, 0, 2, 3)
            frames = frames.unsqueeze(0).to(device)
            
            with torch.no_grad():
                prob = torch.sigmoid(model(frames)).item()
            
            score = int(prob * 10)
            is_anomaly = prob > threshold
            
            print(f"Frame {frame_count}: Score={score}/10 | {'ANOMALY' if is_anomaly else 'Normal'}")
            
            # slide window
            buffer = buffer[8:]
    
    cap.release()

In [39]:
video_path = "/kaggle/input/datasets/ianmolbisht/ritiika/riti.mp4"

In [40]:
import torch
import cv2
import numpy as np
from collections import deque

def run_video_inference(video_path, model, transform, device):
    cap = cv2.VideoCapture(video_path)
    
    buffer = deque(maxlen=8)
    
    model.eval()
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_resized = cv2.resize(frame, (112, 112))
        buffer.append(frame_resized)
        
        if len(buffer) == 8:
            frames = list(buffer)
            
            frames = [transform(f) for f in frames]
            frames = torch.stack(frames)              # (T, C, H, W)
            frames = frames.permute(1, 0, 2, 3)       # (C, T, H, W)
            frames = frames.unsqueeze(0).to(device)   # (1, C, T, H, W)
            
            with torch.no_grad():
                output = model(frames).squeeze()
                prob = torch.sigmoid(output).item()
            
            score = int(prob * 10)
            
            if score <= 3:
                label = "Normal"
            elif score <= 6:
                label = "Suspicious"
            else:
                label = "Anomaly 🚨"
            
            print(f"Score: {score}, {label}")
    
    cap.release()

In [41]:
run_video_inference(video_path, model, val_transform, device)

Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 5, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 4, Suspicious
Score: 3, Normal
Score: 3, Normal
Score: 3, Normal
Score: 4, Suspicious
Score: 4, Suspicious
Scor